# ManuAI

## Preprocess Data

Reads raw audio recordings from `recordings.csv`, splits per-class into **train / val / test** at the *file* level (preventing leakage), chunks each file into 4-second segments (training set with 50 % overlap for data augmentation), and converts every segment into **Kaldi fbank features** as used by the MIT Audio Spectrogram Transformer.

Feature extraction matches `ASTFeatureExtractor` from `preprocessor_config.json` exactly:
- 16 kHz, 128 mel bins, 10 ms frame shift (Kaldi HTK-compat, Hanning window)
- Pad / trim to 1024 time frames (`max_length`)
- Normalise: `(x − mean) / (std × 2)` with AudioSet stats (`mean=−4.2677`, `std=4.5690`)

**Output:** `{data_dir}/processed/{split}/{class}/{file_idx}_{seg_idx}.npy` — shape `(1024, 128)`, float32


## Imports

In [1]:
import os, yaml, random
import numpy as np
import torch
import torchaudio
import pandas as pd
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
import tqdm


## Config

In [2]:
with open('ManuAI.config') as f:
    DATA_DIR = Path(yaml.safe_load(f)['data_directory'])

PROCESSED_DIR = DATA_DIR / 'processed'

# Audio — must be 16 kHz to match ASTFeatureExtractor
SR           = 16000
SEG_DURATION = 4.0
SEG_SAMPLES  = int(SR * SEG_DURATION)   # 64000
TRAIN_STEP   = SEG_SAMPLES // 2         # 32000 — 50 % overlap for train augmentation
EVAL_STEP    = SEG_SAMPLES              # no overlap for val / test

# Kaldi fbank settings — must match preprocessor_config.json / ASTFeatureExtractor defaults
N_MELS      = 128
FRAME_SHIFT = 10            # ms — 100 frames/sec
TARGET_LEN  = 1024          # time frames — matches ASTFeatureExtractor max_length (~10.24 s)

# AST AudioSet normalization statistics (from preprocessor_config.json)
AST_MEAN    = -4.2677393
AST_STD     =  4.5689974
# Resulting shape per segment: (TARGET_LEN, N_MELS) = (1024, 128), float32
# 4-second audio produces ~400 real frames; remaining 624 frames are zero-padded.

# Split
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
TEST_RATIO   = 0.15
RANDOM_STATE = 21

# Set True to overwrite already-processed classes
FORCE_REPROCESS = False

print(f'Data dir:      {DATA_DIR}')
print(f'Processed dir: {PROCESSED_DIR}')
print(f'Segment shape: ({TARGET_LEN}, {N_MELS})')


Data dir:      /Users/harrywills/Desktop/University/AIML339/ManuAI.nosync/data
Processed dir: /Users/harrywills/Desktop/University/AIML339/ManuAI.nosync/data/processed
Segment shape: (1024, 128)


## Discover & Group Files

In [ ]:
recordings_csv = DATA_DIR / 'recordings.csv'
if not recordings_csv.exists():
    raise FileNotFoundError(f'recordings.csv not found at {recordings_csv}. Run data_download.ipynb first.')

df = pd.read_csv(recordings_csv)
df = df[df['label'].notna() & (df['label'] != '')].copy()
df['_path'] = df['filename'].apply(lambda f: DATA_DIR / f)
exists = df['_path'].apply(lambda p: p.exists())

missing = (~exists).sum()
if missing:
    print(f'[warn] {missing} files in recordings.csv not found on disk — skipped')

df = df[exists]
class_files: dict[str, list[Path]] = defaultdict(list)
for label, path in zip(df['label'], df['_path']):
    class_files[label].append(path)

print(f'Found {len(df)} files across {len(class_files)} classes\n')
for cls, files in sorted(class_files.items()):
    print(f'  {cls:<30} {len(files):>4} files')

FileNotFoundError: recordings.csv not found at /Users/harrywills/Desktop/University/AIML339/ManuAI.nosync/data/recordings.csv. Run data_download.ipynb first.

: 

## Per-Class Train / Val / Test Split

Files (not segments) are split to prevent any recording appearing across multiple sets.

In [ ]:
splits: dict[str, dict[str, list[Path]]] = {}

for cls, files in sorted(class_files.items()):
    shuffled = files.copy()
    random.Random(RANDOM_STATE).shuffle(shuffled)
    n = len(shuffled)
    n_test = max(1, round(n * TEST_RATIO))
    n_val  = max(1, round(n * VAL_RATIO))
    splits[cls] = {
        'test':  shuffled[:n_test],
        'val':   shuffled[n_test:n_test + n_val],
        'train': shuffled[n_test + n_val:],
    }

# Print file-level split table
header = f'  {"class":<30} {"train":>6} {"val":>5} {"test":>5} {"total":>6}'
print(header)
print('  ' + '-' * (len(header) - 2))
for cls, s in sorted(splits.items()):
    print(f'{cls:<30} {len(s["train"]):>6} {len(s["val"]):>5} {len(s["test"]):>5} {len(s["train"]) + len(s["val"]) + len(s["test"]):>6}')

## Audio Processing Helpers

In [ ]:
def load_audio(path: Path) -> torch.Tensor:
    """Load audio as mono float32 at SR. Returns (1, n_samples) tensor."""
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SR:
        waveform = torchaudio.functional.resample(waveform, sr, SR)
    return waveform


def chunk_audio(waveform: torch.Tensor, step: int) -> list[torch.Tensor]:
    """Slice waveform into SEG_SAMPLES-length chunks with given step. Zero-pads final chunk."""
    y = waveform.squeeze(0)
    n = len(y)
    if n == 0:
        return []
    chunks = []
    n_start = max(1, n - SEG_SAMPLES + 1)
    for start in range(0, n_start, step):
        seg = y[start:start + SEG_SAMPLES]
        if len(seg) < SEG_SAMPLES:
            seg = torch.nn.functional.pad(seg, (0, SEG_SAMPLES - len(seg)))
        chunks.append(seg.unsqueeze(0))  # (1, SEG_SAMPLES)
    return chunks


def to_fbank(seg: torch.Tensor) -> np.ndarray:
    """Convert a (1, SEG_SAMPLES) audio segment to Kaldi fbank features.

    Matches HuggingFace ASTFeatureExtractor exactly:
      - torchaudio.compliance.kaldi.fbank with htk_compat, hanning window, 10ms frame shift
      - Pad/trim to TARGET_LEN time frames
      - Normalise: (x − AST_MEAN) / (AST_STD × 2)

    Output: (TARGET_LEN, N_MELS) float32 numpy array
    """
    fbank = torchaudio.compliance.kaldi.fbank(
        seg,
        htk_compat=True,
        sample_frequency=SR,
        use_energy=False,
        window_type='hanning',
        num_mel_bins=N_MELS,
        dither=0.0,
        frame_shift=FRAME_SHIFT,
    )  # (n_frames, N_MELS)

    n_frames = fbank.shape[0]
    if n_frames < TARGET_LEN:
        fbank = torch.nn.functional.pad(fbank, (0, 0, 0, TARGET_LEN - n_frames))
    else:
        fbank = fbank[:TARGET_LEN]

    fbank = (fbank - AST_MEAN) / (AST_STD * 2)
    return fbank.numpy().astype(np.float32)


## Generate Kaldi Fbank Features & Save

For each split + class: load audio files, resample to 16 kHz, chunk into 4-second segments, compute Kaldi fbank features (matching `ASTFeatureExtractor`), and save each segment as a `.npy` file.  
Each file is (1024, 128) float32 — ready to pass directly to `ASTForAudioClassification`.


In [ ]:
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

def process_file(fpath: Path, step: int, out_dir: Path, file_idx: int) -> tuple[int, str | None]:
    """Load one audio file, chunk, compute Kaldi fbank, save as .npy. Returns (n_saved, error)."""
    try:
        waveform, sr = torchaudio.load(fpath)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != SR:
            waveform = torchaudio.functional.resample(waveform, sr, SR)
    except Exception as e:
        return 0, f'Could not load {fpath.name}: {e}'

    # DC removal — matches original AST dataloader (waveform - waveform.mean())
    waveform = waveform - waveform.mean()
    y = waveform.squeeze(0)
    n = len(y)
    if n == 0:
        return 0, f'Empty audio: {fpath.name}'

    saved = 0
    n_start = max(1, n - SEG_SAMPLES + 1)
    for seg_idx, start in enumerate(range(0, n_start, step)):
        seg = y[start:start + SEG_SAMPLES]
        if len(seg) < SEG_SAMPLES:
            seg = torch.nn.functional.pad(seg, (0, SEG_SAMPLES - len(seg)))
        seg = seg.unsqueeze(0)  # (1, SEG_SAMPLES)

        # Kaldi fbank — matches ASTFeatureExtractor exactly
        fbank = torchaudio.compliance.kaldi.fbank(
            seg,
            htk_compat=True,
            sample_frequency=SR,
            use_energy=False,
            window_type='hanning',
            num_mel_bins=N_MELS,
            dither=0.0,
            frame_shift=FRAME_SHIFT,
        )  # (n_frames, N_MELS)

        # Pad / trim to TARGET_LEN
        n_frames = fbank.shape[0]
        if n_frames < TARGET_LEN:
            fbank = torch.nn.functional.pad(fbank, (0, 0, 0, TARGET_LEN - n_frames))
        else:
            fbank = fbank[:TARGET_LEN]

        # AST-style normalisation: (x − mean) / (std × 2)
        fbank = (fbank - AST_MEAN) / (AST_STD * 2)

        np.save(out_dir / f'{file_idx:06d}_{seg_idx:04d}.npy', fbank.numpy().astype(np.float32))
        saved += 1

    return saved, None


SPLIT_STEPS = [('train', TRAIN_STEP), ('val', EVAL_STEP), ('test', EVAL_STEP)]

for split_name, step in SPLIT_STEPS:
    out_base = PROCESSED_DIR / split_name
    out_base.mkdir(parents=True, exist_ok=True)

    class_iter = tqdm.tqdm(sorted(splits.keys()), desc=f'{split_name:<5}', unit='class')
    for cls in class_iter:
        cls_dir = out_base / cls

        if cls_dir.exists() and any(cls_dir.iterdir()) and not FORCE_REPROCESS:
            class_iter.set_postfix(status=f'{cls} skipped')
            continue

        cls_dir.mkdir(exist_ok=True)
        file_paths = splits[cls][split_name]
        cls_segs = 0

        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            futures = [ex.submit(process_file, fp, step, cls_dir, i) for i, fp in enumerate(file_paths)]
            for fut in futures:
                n_saved, err = fut.result()
                if err:
                    print(f'[warn] {err}')
                cls_segs += n_saved

        class_iter.set_postfix(status=f'{cls} → {cls_segs} segs')

print('Done.')


## Summary

In [ ]:
print(f'\n  {"class":<30} {"train":>7} {"val":>7} {"test":>7} {"total":>8}')
print('  ' + '-' * 63)

total_segs  = 0
total_bytes = 0

for cls in sorted(splits.keys()):
    counts = {}
    for split_name in ('train', 'val', 'test'):
        cls_dir = PROCESSED_DIR / split_name / cls
        if cls_dir.exists():
            npys = list(cls_dir.glob('*.npy'))
            counts[split_name] = len(npys)
            total_segs  += len(npys)
            total_bytes += sum(p.stat().st_size for p in npys)
        else:
            counts[split_name] = 0
    row_total = sum(counts.values())
    print(f'  {cls:<30} {counts["train"]:>7} {counts["val"]:>7} {counts["test"]:>7} {row_total:>8}')

print('  ' + '-' * 63)
print(f'  {"TOTAL":<30} {"":>7} {"":>7} {"":>7} {total_segs:>8}')
print(f'\n  Disk usage: {total_bytes / 1e9:.2f} GB')


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

split_to_preview = 'train'
class_names = sorted(splits.keys())
if not class_names:
    raise ValueError('No classes available to display.')

n_cols = 3
n_rows = math.ceil(len(class_names) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, cls in zip(axes, class_names):
    cls_dir = PROCESSED_DIR / split_to_preview / cls
    npy_files = sorted(cls_dir.glob('*.npy')) if cls_dir.exists() else []

    if not npy_files:
        ax.set_title(f'{cls}\n(no samples)')
        ax.axis('off')
        continue

    spec = np.load(npy_files[0])
    display_spec = spec * (AST_STD * 2) + AST_MEAN

    ax.imshow(display_spec.T, origin='lower', aspect='auto', cmap='magma')
    ax.set_title(cls)
    ax.set_xlabel('time frames')
    ax.set_ylabel('mel bins')

for ax in axes[len(class_names):]:
    ax.axis('off')

plt.suptitle(f'Per-class AST feature preview ({split_to_preview})', y=1.02)
plt.tight_layout()
plt.show()
